In [0]:
# 01 - Importar funções e carregar a tabela Silver

from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver = spark.table(
    "workspace.projeto_integrador.silver_exames_citopatologicos"
)

display(df_silver.limit(10))


In [0]:
# 02 - Criar base Gold com os anos completos de 2016 a 2025

df_gold = (
    df_silver
    .filter(
        (F.col("ano") >= 2016) &
        (F.col("ano") <= 2025)
    )
    .select(
        "codigo_ibge",
        "municipio",
        "uf",
        "estado",
        "codigo_regiao_brasil",
        "regiao_brasil",
        "sigla_regiao_brasil",
        "codigo_regiao_saude",
        "regiao_saude",
        "codigo_macrorregiao",
        "macrorregiao",
        "ano",
        "ano_mes",
        "competencia",
        "valor_indicador_municipio"
    )
    .withColumn(
        "teve_exames",
        F.when(
            F.col("valor_indicador_municipio") > 0,
            1
        ).otherwise(0)
    )
)

display(df_gold.limit(10))


In [0]:
# 03 - Salvar a Gold de indicadores municipais

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.projeto_integrador.gold_indicadores_municipais"
    )

print("Tabela gold_indicadores_municipais salva com sucesso.")

In [0]:
# 04 - Definir municípios elegíveis usando somente o período de treinamento

municipios_elegiveis = (
    df_gold
    .filter(
        (F.col("ano") >= 2016) &
        (F.col("ano") <= 2023)
    )
    .groupBy("codigo_ibge")
    .agg(
        F.sum(
            F.when(
                F.col("valor_indicador_municipio") > 0,
                1
            ).otherwise(0)
        ).alias("anos_com_exames")
    )
    .filter(
        F.col("anos_com_exames") == 8
    )
    .select("codigo_ibge")
)

print(
    "Municípios elegíveis usando somente 2016-2023:",
    municipios_elegiveis.count()
)

In [0]:
# 05 - Criar base para Machine Learning sem vazamento temporal

df_ml_base = (
    df_gold
    .join(
        municipios_elegiveis,
        on="codigo_ibge",
        how="inner"
    )
)

print(
    "Quantidade de registros da base:",
    df_ml_base.count()
)

print(
    "Quantidade de municípios:",
    df_ml_base
    .select("codigo_ibge")
    .distinct()
    .count()
)

display(
    df_ml_base
    .orderBy("codigo_ibge", "ano")
    .limit(30)
)

In [0]:
# 06 - Criar variáveis históricas para Machine Learning

janela_municipio = (
    Window
    .partitionBy("codigo_ibge")
    .orderBy("ano")
)

janela_3_anos = (
    Window
    .partitionBy("codigo_ibge")
    .orderBy("ano")
    .rowsBetween(-3, -1)
)

df_ml_features = (
    df_ml_base

    .withColumn(
        "valor_ano_anterior",
        F.lag(
            "valor_indicador_municipio",
            1
        ).over(janela_municipio)
    )

    .withColumn(
        "valor_2_anos_antes",
        F.lag(
            "valor_indicador_municipio",
            2
        ).over(janela_municipio)
    )

    .withColumn(
        "valor_3_anos_antes",
        F.lag(
            "valor_indicador_municipio",
            3
        ).over(janela_municipio)
    )

    .withColumn(
        "media_3_anos_anteriores",
        F.avg(
            "valor_indicador_municipio"
        ).over(janela_3_anos)
    )

    .withColumn(
        "diferenca_ultimo_ano",
        F.col("valor_ano_anterior") -
        F.col("valor_2_anos_antes")
    )
)

display(
    df_ml_features
    .orderBy("codigo_ibge", "ano")
    .limit(30)
)

In [0]:
# 07 - Criar base final de Machine Learning

df_ml_final = (
    df_ml_features
    .filter(
        (F.col("ano") >= 2019) &
        (F.col("ano") <= 2025)
    )
    .filter(
        F.col("valor_ano_anterior").isNotNull() &
        F.col("valor_2_anos_antes").isNotNull() &
        F.col("valor_3_anos_antes").isNotNull() &
        F.col("media_3_anos_anteriores").isNotNull()
    )
)

print(
    "Registros disponíveis para ML:",
    df_ml_final.count()
)

print(
    "Quantidade de municípios:",
    df_ml_final
    .select("codigo_ibge")
    .distinct()
    .count()
)

display(
    df_ml_final
    .orderBy("codigo_ibge", "ano")
    .limit(30)
)

In [0]:
# 08 - Conferir distribuição temporal da base de Machine Learning

distribuicao_temporal = (
    df_ml_final
    .groupBy("ano")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct(
            "codigo_ibge"
        ).alias("municipios")
    )
    .orderBy("ano")
)

display(distribuicao_temporal)

In [0]:
# 09 - Salvar a Gold preparada para Machine Learning

df_ml_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.projeto_integrador.gold_base_ml"
    )

print("Tabela gold_base_ml salva com sucesso.")

In [0]:
# 10 - Validar as tabelas Gold criadas

gold_indicadores = spark.table(
    "workspace.projeto_integrador.gold_indicadores_municipais"
)

gold_ml = spark.table(
    "workspace.projeto_integrador.gold_base_ml"
)

print(
    "Registros Gold Indicadores:",
    gold_indicadores.count()
)

print(
    "Registros Gold ML:",
    gold_ml.count()
)

print(
    "Municípios Gold ML:",
    gold_ml
    .select("codigo_ibge")
    .distinct()
    .count()
)

display(
    gold_ml
    .orderBy("codigo_ibge", "ano")
    .limit(20)
)